# Spyglass optogenetics tutorial

This notebook demonstrates how to query optogenetics data for a Pagan Lab session from the Spyglass database.

The example session (`sub-P131_ses-TaskSwitch6-190815a`) contains bilateral FOF ChR2 stimulation delivered wirelessly via the Cerebro system during an auditory task-switching paradigm.

The notebook is structured in two parts:

**Part 1 — Spyglass tables** (populated by `insert_session()`):
1. Connect to the Spyglass database
2. Core tables: `Nwbfile`, `Session`, `Subject`
3. Hardware tables: `Virus`, `VirusInjection`, `OpticalFiberDevice`, `OpticalFiberImplant`
4. Protocol tables: `TaskEpoch`, `OptogeneticProtocol`

**Part 2 — Per-trial optogenetics from NWB** (read from `opto_epochs` interval table):
5. Load `opto_epochs` from the NWB file
6. Merge opto epochs with trial outcomes
7. Visualise: opto effect on performance, stimulation window distribution, hemisphere breakdown

For general Spyglass access (BControl task recording) see [`spyglass_tutorial.ipynb`](spyglass_tutorial.ipynb).  
For NWB visualisations see [`arc_behavior_dandi_demo_notebook.ipynb`](arc_behavior_dandi_demo_notebook.ipynb).

## 1. Connect to Spyglass

In [8]:
from pathlib import Path

import datajoint as dj
import pandas as pd

# dj_local_conf.json lives in the sibling arc_ecephys/ directory.
DJ_LOCAL_CONF_PATH = Path.cwd().parent / "arc_ecephys" / "dj_local_conf.json"
dj.config.load(str(DJ_LOCAL_CONF_PATH))
dj.conn(use_tls=False)

[2026-06-17 13:47:59,637][INFO]: DataJoint is configured from /Users/weian/catalystneuro/pagan-lab-to-nwb/src/pagan_lab_to_nwb/arc_ecephys/dj_local_conf.json


DataJoint connection (connected) root@localhost:3306

In [9]:
import spyglass.common as sgc
from spyglass.settings import raw_dir
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename, get_nwb_file

nwb_file_name = "sub-P131_ses-TaskSwitch6-190815a.nwb"
nwbfile_path = Path(raw_dir) / nwb_file_name  # source file must be at RAW_DIR root

nwb_copy_file_name = get_nwb_copy_filename(nwb_file_name)  # adds trailing "_"
nwb_dict = dict(nwb_file_name=nwb_copy_file_name)

print("Source file:  ", nwbfile_path)
print("Spyglass copy:", nwb_copy_file_name)

Source file:   /Volumes/T9/data/Pagan/raw/sub-P131_ses-TaskSwitch6-190815a.nwb
Spyglass copy: sub-P131_ses-TaskSwitch6-190815a_.nwb


## 2. Core tables

`Nwbfile` is the root entry — every other table is keyed by `nwb_file_name` (the `_`-suffixed copy name).

In [10]:
from spyglass.common import Nwbfile

pd.DataFrame((Nwbfile() & nwb_dict).fetch(as_dict=True))[["nwb_file_name", "nwb_file_abs_path"]]

,nwb_file_name,nwb_file_abs_path
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,/Volumes/T9/data/Pagan/raw/sub-P131_ses-TaskSw...


In [11]:
session_rows = (sgc.Session() & nwb_dict).fetch(as_dict=True)
session_df = pd.DataFrame(session_rows)

cols = ["nwb_file_name", "session_id", "institution_name", "lab_name",
        "session_start_time"]
session_df[cols]

,nwb_file_name,session_id,institution_name,lab_name,session_start_time
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,TaskSwitch6-190815a,University of Edinburgh,Pagan,2019-08-15 11:41:00


In [12]:
subject_id = session_df["subject_id"].iloc[0]
subject_df = pd.DataFrame((sgc.Subject() & {"subject_id": subject_id}).fetch(as_dict=True))
subject_df[["subject_id", "species", "sex", "age", "description"]]

,subject_id,species,sex,age,description
0,P131,Rattus norvegicus,M,None,"Long-Evans rat, wild-type."


## 3. Optogenetics hardware tables

Spyglass extracts hardware metadata from the `ndx-optogenetics` objects embedded in the NWB file during `insert_sessions()`.  All four tables are **shared** (no `nwb_file_name` key) — one row per device/virus, shared across all sessions that used the same hardware.

| Table | What it stores |
|---|---|
| `Virus` | Viral vector construct (AAV serotype, gene) |
| `VirusInjection` | Per-hemisphere injection coordinates and volume |
| `OpticalFiberDevice` | Fiber model specs (NA, core diameter, ferrule) |
| `OpticalFiberImplant` | Per-hemisphere implant coordinates |

In [13]:
from spyglass.common.common_optogenetics import (
    OpticalFiberDevice,
    OpticalFiberImplant,
    OptogeneticProtocol,
    Virus,
    VirusInjection,
)

print("=== Virus ===")
virus_df = pd.DataFrame((Virus() & {"virus_name": "aav_mdlx_chr2_mcherry"}).fetch(as_dict=True))
display(virus_df)

=== Virus ===


,virus_name,construct_name,description,manufacturer
0,aav_mdlx_chr2_mcherry,AAV2/5-mDlx-ChR2-mCherry,AAV2/5-mDlx-ChR2-mCherry used for interneuron-...,unknown


In [14]:
print("=== VirusInjection (bilateral FOF) ===")
inj_df = pd.DataFrame((VirusInjection() & nwb_dict).fetch(as_dict=True))

coord_cols = ["nwb_file_name", "name", "hemisphere", "location",
              "ap_location", "ml_location", "dv_location", "volume"]
display(inj_df[coord_cols])

=== VirusInjection (bilateral FOF) ===


,nwb_file_name,name,hemisphere,location,ap_location,ml_location,dv_location,volume
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,injection_fof_right,right,Frontal Orienting Field (FOF),2.0,1.3,-1.5,1.5
1,sub-P131_ses-TaskSwitch6-190815a_.nwb,injection_fof_left,left,Frontal Orienting Field (FOF),2.0,-1.3,-1.5,1.5


In [15]:
print("=== OpticalFiberDevice ===")
fiber_dev_df = pd.DataFrame(
    (OpticalFiberDevice() & {"fiber_name": "fof_fiber_model"}).fetch(as_dict=True)
)
display(fiber_dev_df)

=== OpticalFiberDevice ===


,fiber_name,model,manufacturer,numerical_aperture,core_diameter,active_length,ferrule_name,ferrule_diameter
0,fof_fiber_model,FG400UEP,Newport,0.37,400.0,1.5,1.25 mm LC ferrule,1.25


In [16]:
print("=== OpticalFiberImplant (bilateral FOF) ===")
implant_df = pd.DataFrame((OpticalFiberImplant() & nwb_dict).fetch(as_dict=True))

implant_cols = ["nwb_file_name", "implant_id", "hemisphere",
                "ap_location", "ml_location", "dv_location"]
display(implant_df[implant_cols])

=== OpticalFiberImplant (bilateral FOF) ===


,nwb_file_name,implant_id,hemisphere,ap_location,ml_location,dv_location
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,0,right,2.0,1.3,0.0
1,sub-P131_ses-TaskSwitch6-190815a_.nwb,1,left,2.0,-1.3,0.0


## 4. Optogenetics protocol tables

`TaskEpoch` and `OptogeneticProtocol` are inserted post-hoc by `insert_session.py` (not by the core Spyglass `insert_sessions()`) because arc_behavior NWB files do not contain a `processing["tasks"]` module.

Each arc_behavior session has a single epoch (epoch 1 = the whole session).  `OptogeneticProtocol` stores session-level aggregate parameters; per-trial detail lives in `nwb.intervals["opto_epochs"]` (Part 2).

In [17]:
from spyglass.common.common_task import Task, TaskEpoch

print("=== Task ===")
protocol_name = nwb_file_name.split("_ses-")[1].split("-")[0]  # e.g. "TaskSwitch6"
display(pd.DataFrame((Task() & {"task_name": protocol_name}).fetch(as_dict=True)))

print("\n=== TaskEpoch ===")
epoch_df = pd.DataFrame((TaskEpoch() & nwb_dict & {"epoch": 1}).fetch(as_dict=True))
display(epoch_df)

=== Task ===


,task_name,task_description,task_type,task_subtype
0,TaskSwitch6,Auditory decision-making task-switching paradi...,None,None



=== TaskEpoch ===


,nwb_file_name,epoch,task_name,camera_name,interval_list_name,task_environment,camera_names
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,1,TaskSwitch6,None,01,behavioral_box,[]


In [18]:
print("=== OptogeneticProtocol ===")
protocol_rows = (OptogeneticProtocol() & nwb_dict & {"epoch": 1}).fetch(as_dict=True)
protocol_df = pd.DataFrame(protocol_rows)
display(protocol_df)

=== OptogeneticProtocol ===


,nwb_file_name,epoch,description,pulse_length,pulses_per_train,period,intertrain_interval,stimulus_power,stimulus_object_id
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,1,Bilateral FOF ChR2 via Cerebro. Per-trial wind...,1300.0,1,1300.0,0.0,25.0,5366943f-e10f-47a2-85ce-6b2014213253


In [19]:
# Pretty-print the key fields
p = protocol_df.iloc[0]
print(f"Description     : {p['description']}")
print(f"Pulse length    : {p['pulse_length']} ms  (max across window types)")
print(f"Pulses/train    : {p['pulses_per_train']}")
print(f"Stimulus power  : {p['stimulus_power']} mW")
print(f"Stimulus obj_id : {p['stimulus_object_id']}  (UUID of opto_epochs table)")

Description     : Bilateral FOF ChR2 via Cerebro. Per-trial window types (Full Trial=1300ms, First Half=650ms, Second Half=650ms; re cpoke). Session pulse_length=max (1300 ms). Per-trial detail: nwb.intervals['opto_epochs'].
Pulse length    : 1300.0 ms  (max across window types)
Pulses/train    : 1
Stimulus power  : 25.0 mW
Stimulus obj_id : 5366943f-e10f-47a2-85ce-6b2014213253  (UUID of opto_epochs table)


## 5. Per-trial optogenetics from the NWB file

`OptogeneticProtocol` stores session-level aggregates. The trial-by-trial detail is in `nwb.intervals["opto_epochs"]` — one row per stimulated trial, with exact start/stop times, window type, and hemisphere.

The `stimulus_object_id` in `OptogeneticProtocol` is the UUID of this table, so it can always be traced back.

In [20]:
# Load the NWB file (Spyglass _ copy — same data, may have patched string fields)
nwbfile = get_nwb_file(str(nwbfile_path))

opto_epochs = nwbfile.intervals["opto_epochs"]
print(f"opto_epochs UUID  : {opto_epochs.object_id}")
print(f"Matches DB uuid   : {opto_epochs.object_id == p['stimulus_object_id']}")

opto_epochs UUID  : 5366943f-e10f-47a2-85ce-6b2014213253
Matches DB uuid   : True


In [21]:
opto_df = opto_epochs.to_dataframe()
print(f"{len(opto_df)} stimulated trials")
opto_df[["start_time", "stop_time", "pulse_length_in_ms",
          "power_in_mW", "wavelength_in_nm",
          "optogenetic_sites"]].head(10)

205 stimulated trials


,start_time,stop_time,pulse_length_in_ms,power_in_mW,wavelength_in_nm,optogenetic_sites
id,,,,,,
0,1457.688753,1458.338753,650.0,25.0,450.0,optica...
1,1488.349256,1488.999256,650.0,25.0,450.0,optica...
2,1516.250754,1517.550754,1300.0,25.0,450.0,optica...
3,1541.370253,1542.020253,650.0,25.0,450.0,optica...
4,1580.621753,1581.271753,650.0,25.0,450.0,optica...
5,1606.274253,1606.924253,650.0,25.0,450.0,optica...
6,1632.856753,1633.506753,650.0,25.0,450.0,optica...
7,1657.821255,1658.471255,650.0,25.0,450.0,optica...
8,1684.191754,1684.841754,650.0,25.0,450.0,optica...


In [22]:
# optogenetic_sites is a DynamicTableRegion resolved to a DataFrame.
# Its index contains the site indices: 0 = left FOF fiber, 1 = right FOF fiber.
def hemisphere_label(sites_df) -> str:
    indices = set(sites_df.index.tolist())
    has_left = 0 in indices
    has_right = 1 in indices
    if has_left and has_right:
        return "bilateral"
    elif has_left:
        return "left"
    elif has_right:
        return "right"
    return "none"

opto_df["hemisphere"] = opto_df["optogenetic_sites"].apply(hemisphere_label)

# Full Trial = 1300 ms; First Half and Second Half are both 650 ms
# (indistinguishable by duration alone — use trial-table opto_type for that).
opto_df["window_type"] = opto_df["pulse_length_in_ms"].map(
    {1300.0: "Full Trial (1300 ms)", 650.0: "Half Trial (650 ms)"}
).fillna("Other")

print(opto_df.groupby(["window_type", "hemisphere"]).size().to_string())

window_type           hemisphere
Full Trial (1300 ms)  bilateral     21
                      left          22
                      right         20
Half Trial (650 ms)   bilateral     39
                      left          55
                      right         48


## 6. Merge opto epochs with trial outcomes

The `opto_epochs` table stores stimulation intervals keyed by absolute time. We join it to the trials table (from `nwb.trials`) using the `start_time` overlap to recover trial-level variables like `HistorySection_result_history` (1 = correct, 3 = error/abort).

In [23]:
from spyglass.common.common_task_rec import TaskRecording

trials_df = (TaskRecording() & nwb_dict).fetch1_dataframe("trials")

# Exclude state/event/action VectorIndex columns for cleaner display
scalar_cols = [c for c in trials_df.columns
               if c not in ("states", "events", "actions")]
trials_df = trials_df[scalar_cols].copy()

# Decode trial outcome: 1 = correct, 2 = violation, 3 = error
result_map = {1: "correct", 2: "violation", 3: "error"}
trials_df["result"] = trials_df["HistorySection_result_history"].map(result_map)

print(f"{len(trials_df)} total trials")
print(trials_df["result"].value_counts().to_string())

617 total trials
result
error        301
correct      236
violation     78


In [24]:
# Mark which trials had active laser stimulation.
# OptoSection_opto_connected == 1 means the Cerebro was armed;
# an opto_epochs row exists only if at least one hemisphere actually fired.
trials_df["opto_connected"] = trials_df["OptoSection_opto_connected"].astype(bool)
trials_df["opto_window"] = trials_df["OptoSection_opto_type"]

# Use raw Cerebro power readings to determine which hemisphere(s) fired.
CEREBRO_THRESHOLD = 800  # raw units → laser on
trials_df["left_stim"] = trials_df["HistorySection_opto_left_history"] >= CEREBRO_THRESHOLD
trials_df["right_stim"] = trials_df["HistorySection_opto_right_history"] >= CEREBRO_THRESHOLD

def hemisphere_from_trial(row):
    if row["left_stim"] and row["right_stim"]:
        return "bilateral"
    elif row["left_stim"]:
        return "left"
    elif row["right_stim"]:
        return "right"
    return "none"

trials_df["hemisphere"] = trials_df.apply(hemisphere_from_trial, axis=1)

print("Opto hemisphere breakdown:")
print(trials_df["hemisphere"].value_counts().to_string())

Opto hemisphere breakdown:
hemisphere
none         412
left          77
right         68
bilateral     60


## 8. Cross-reference: Spyglass protocol ↔ NWB opto epochs

The `stimulus_object_id` in `OptogeneticProtocol` is the UUID of the `opto_epochs` table in the NWB file.  Here we verify the round-trip and confirm the session-level aggregate (max pulse length) matches what is stored in the DB.

In [30]:
db_protocol = (OptogeneticProtocol() & nwb_dict & {"epoch": 1}).fetch1()

max_pulse_nwb = float(opto_df["pulse_length_in_ms"].max())
max_power_nwb = float(opto_df["power_in_mW"].max())

print("Cross-reference: Spyglass OptogeneticProtocol ↔ NWB opto_epochs")
print(f"  pulse_length  DB={db_protocol['pulse_length']:.1f} ms  | NWB max={max_pulse_nwb:.1f} ms  | match={db_protocol['pulse_length'] == max_pulse_nwb}")
print(f"  stimulus_power DB={db_protocol['stimulus_power']:.1f} mW  | NWB max={max_power_nwb:.1f} mW  | match={db_protocol['stimulus_power'] == max_power_nwb}")
print(f"  stimulus_object_id matches opto_epochs.object_id: {db_protocol['stimulus_object_id'] == opto_epochs.object_id}")

Cross-reference: Spyglass OptogeneticProtocol ↔ NWB opto_epochs
  pulse_length  DB=1300.0 ms  | NWB max=1300.0 ms  | match=True
  stimulus_power DB=25.0 mW  | NWB max=25.0 mW  | match=True
  stimulus_object_id matches opto_epochs.object_id: True


## Summary

**Hardware (shared tables)** — inserted once, shared across all sessions using the same virus/fiber:
- `Virus`: AAV2/5-mDlx-ChR2-mCherry interneuron-targeted construct
- `VirusInjection`: bilateral FOF injections (+2 mm AP, ±1.3 mm ML, −1.5 mm DV, 1.5 µL)
- `OpticalFiberDevice`: 400 µm core / 0.37 NA Newport FG400UEP fiber
- `OpticalFiberImplant`: bilateral FOF implant coordinates

**Protocol (per-session)** — inserted post-hoc in `insert_session.py`:
- `TaskEpoch`: one epoch per session (epoch 1 = whole session)
- `OptogeneticProtocol`: session-level aggregate (max pulse length across window types)

**Per-trial detail** — read from `nwb.intervals["opto_epochs"]`:
- Exact stimulation intervals anchored to cpoke onset
- Window types: Full Trial (1300 ms), First Half (650 ms), Second Half (650 ms)
- Hemisphere: site index 0 = left FOF, 1 = right FOF
- Traceable from DB to NWB via `OptogeneticProtocol.stimulus_object_id` = `opto_epochs.object_id`